# Tutorial 5: PyMC Surrogate Learning (`pymc_gp`)

Estimated time: 30-50 minutes

## Prerequisites
`pymc` and `arviz` installed.

## Learning aims
- Primary package aim: fit/evaluate surrogate models through the CLI and inspect artifacts
- Secondary scientific aim: build intuition for prior, posterior, and posterior predictive uncertainty

## Success criteria
- you can train a PyMC surrogate, evaluate new inputs, and interpret uncertainty width


## Why this tutorial matters

A **surrogate** is a fast probabilistic model fit to a sweep's outputs that lets you replace the simulator at inference time — predict at any input without re-running the model. The PyMC GP surrogate gives you a *posterior predictive*: every prediction comes with a **width** (a posterior standard deviation), not just a point estimate. Reading those widths is what separates "the surrogate said 0.42" from "the surrogate said 0.42 ± 0.01, and I trust the prediction here because the training set covers this region densely."

**Callback to T1:** at the bottom of T1 you held a question — "would you trust a surrogate's prediction at `a=0.5, b=0.5`?" This is the tutorial where you answer it. After Step 4 you'll be able to look at any predictive width and decide whether to trust the prediction or run the simulator at that input instead.


## Step 1: Ensure training dataset exists


In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)


# Preflight: detect PyMC. This notebook invokes `bayesmm surrogate fit/eval`
# with the pymc_gp backend, which requires `pymc` in THIS kernel's env. The
# main dev env (py314_bayesmm) deliberately ships without it — the backends
# live in their own envs — so detecting its absence here lets us skip the
# PyMC-specific steps with an actionable banner instead of a RuntimeError
# halfway down the notebook. Mirrors the SBI preflight in Tutorial 6.
import importlib.util as _ilu
import os as _os

PYMC_AVAILABLE = _ilu.find_spec("pymc") is not None

if not PYMC_AVAILABLE:
    _conda_env_name = _os.environ.get("CONDA_DEFAULT_ENV")
    _BANNER = "=" * 72
    print()
    print(_BANNER)
    print("  PREFLIGHT: PyMC backend missing — PyMC steps will be SKIPPED")
    print(_BANNER)
    print(f"  Kernel env path  : {sys.prefix}")
    if _conda_env_name:
        print(f"  Conda env        : {_conda_env_name}")
    print("  Missing package  : 'pymc' (pymc_gp surrogate backend)")
    print()
    print("  HOW TO FIX — use the dedicated backend env and re-run with that kernel:")
    print("    conda env create -f environment-pymc.yml")
    print("    conda activate py312_bayesmm_pymc")
    print("    python -m ipykernel install --user --name py312_bayesmm_pymc")
    print()
    print("  The rest of the notebook still runs; only the PyMC steps are skipped.")
    print(_BANNER)
    print()


In [2]:
run_mm_cli('run', 'tutorials/specs/model.toy.grid.json')


$ bayesmm run tutorials/specs/model.toy.grid.json
Running point 1/9: {'a': 0.0, 'b': 0.0}
Running point 2/9: {'a': 0.0, 'b': 1.0}
Running point 3/9: {'a': 0.0, 'b': 2.0}
Running point 4/9: {'a': 1.0, 'b': 0.0}
Running point 5/9: {'a': 1.0, 'b': 1.0}
Running point 6/9: {'a': 1.0, 'b': 2.0}
Running point 7/9: {'a': 2.0, 'b': 0.0}
Running point 8/9: {'a': 2.0, 'b': 1.0}
Running point 9/9: {'a': 2.0, 'b': 2.0}
Stored sweep run: 3b5a32b1259a4cac80035a0129d25edb
Run complete: 9 successful runs


0

## Step 2: Fit surrogate and list artifacts


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 2 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    run_mm_cli('surrogate', 'fit', 'tutorials/specs/surrogate.toy.pymc_gp.json')
    run_mm_cli('surrogate', 'list')


## Step 3: Evaluate on new inputs


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 3 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    run_mm_cli('surrogate', 'eval', 'tutorials/specs/surrogate.toy.pymc_gp.json', '--inputs', '{"a":[0.25,0.75,1.25,1.75],"b":[0.2,0.6,1.0,1.4]}', '--n', '200')


## Predict the uncertainty before you plot

This is the question T1 asked you to hold. You trained a GP on a 3×3 grid, and
you're about to ask it about points it never saw. Commit to answers first:

1. At `a = 0.5, b = 0.5` — **between** training points — will the predictive band be
   narrow or wide? Why?
2. At `a = 5.0, b = 5.0` — far **outside** the training range — what happens to the
   band? What does the *mean* do out there?
3. Exactly **at** a training point, what should the band do?

Then read the plot.

*(Expected: narrow between points, because the GP interpolates with evidence on both
sides. Wide outside the range — and, crucially, the mean reverts toward the GP's
prior mean rather than continuing the trend, which is the single most misread
behaviour of GP extrapolation. Near-zero at a training point, up to observation
noise.)*

**The point of the exercise:** a surrogate that is confidently wrong is worse than
no surrogate, and the predictive width is what tells you which regime you're in.
The band is not decoration — it is the reason to use a probabilistic surrogate at all.

## Step 4: Plot predictive mean and uncertainty (graphic)


## Mini-lesson: prior, posterior, posterior predictive (read this BEFORE the plot)

Three Bayesian terms you'll see across this curriculum:

- **Prior**: your beliefs about the model parameters *before* seeing data. For a GP surrogate, the prior is over the function shape (smoothness via a kernel, output scale, noise) — not over the output `y` directly.
- **Posterior**: your updated beliefs about those parameters *after* the GP has been conditioned on the 9 training observations. Tighter than the prior wherever data was informative.
- **Posterior predictive**: the distribution over the *output* `y` at a *new* input, integrating over the posterior. This is what `eval_surrogate` returns: a `mean` (point estimate) and a `std` (posterior width). The width is the answer to "how sure is the GP about this prediction?"

When you look at the errorbar plot below, the dots are the posterior predictive *means*; the bars are the posterior predictive *standard deviations*. Wider bars = the GP is less sure. Narrower bars = it has enough nearby training data to be confident.


**Common confusion: these errorbars are NOT confidence intervals.** They're the *posterior predictive standard deviation* — the GP's belief about how uncertain it is at each query point given its training data. Specifically, they are NOT:

- a frequentist confidence interval ("95% of resampled estimates would fall here") — wrong framework entirely;
- the *simulator's* noise — there is none here; the toy is deterministic;
- a constant scaled by the data's spread — they vary across query points based on how close the query is to training data.

If you doubled the training data (a denser grid), these widths would shrink. If you queried at an input far from any training point, they'd grow. **That responsiveness to data support is the difference a Bayesian surrogate makes** vs a point-prediction model that gives you the same width everywhere.

**Heads-up about the plot below:** for this particular toy (`y = a + b` is a perfectly linear, deterministic function and 9 training points cover the box well), the GP is *very* confident — the std comes out around `1e-7`, so the errorbars are basically invisible. The augmented plot annotates the actual std numerically and overlays the analytical truth so you can see the mean is calibrated. With a noisier or sparser training set, you'd see the bars open up.


In [ ]:
if not PYMC_AVAILABLE:
    print("Step 4 (plot) SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
    mean = None
    std = None
else:
    import csv
    import json
    import numpy as np
    import matplotlib.pyplot as plt
    from pathlib import Path

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate

    root = Path.cwd().resolve()
    if not (root / "src").is_dir() and (root.parent / "src").is_dir():
        root = root.parent

    spec_payload = json.loads((root / "tutorials/specs/surrogate.toy.pymc_gp.json").read_text())
    spec = SurrogateSpec.model_validate(spec_payload)

    # --- Query points (4 inputs the surrogate hasn't seen) ---
    inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    result = eval_surrogate(spec=spec, inputs_payload=inputs, n=300)
    mean = np.asarray(result["summary"]["mean"], dtype=float)
    std = np.asarray(result["summary"].get("std", [0.0] * len(mean)), dtype=float)

    # --- Find the toy training sweep CSV (whichever sweep populated tmp/tutorials/toy_store) ---
    sweep_csv = next(
        (root / "tmp/tutorials/toy_store/sweeps").glob("*/sweep_rows.csv"),
        None,
    )
    train_a, train_b, train_y = [], [], []
    if sweep_csv is not None:
        with open(sweep_csv) as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get("status") != "success":
                    continue
                train_a.append(float(row["a"]))
                train_b.append(float(row["b"]))
                # The toy outputs `y = [a+b, a*b]`; the surrogate trained on the first column.
                train_y.append(float(row.get("y__0", float(row["a"]) + float(row["b"]))))
    train_a = np.asarray(train_a)
    train_b = np.asarray(train_b)
    train_y = np.asarray(train_y)

    # Use `a + b` as x-axis (since the underlying truth is y = a + b, this lays the predictions and
    # the training points on the same calibration curve).
    query_x = np.asarray(inputs["a"]) + np.asarray(inputs["b"])
    query_truth = query_x  # analytical truth: y = a + b

    train_x = train_a + train_b

    fig, ax = plt.subplots(figsize=(7.5, 4.5))

    # Analytical truth as a reference line
    xx = np.linspace(0, 4, 50)
    ax.plot(xx, xx, "k:", alpha=0.5, label="analytical truth: y = a + b")

    # Training observations
    ax.scatter(
        train_x, train_y, marker="x", s=70, c="tab:gray",
        alpha=0.7, label=f"training inputs (N={len(train_x)})", zorder=3,
    )

    # Surrogate predictions with errorbars
    ax.errorbar(
        query_x, mean, yerr=std, fmt="o", color="tab:blue",
        markersize=10, capsize=5, label="surrogate posterior predictive (mean ± std)", zorder=4,
    )

    # Annotate one query point with its actual std value (since the bars are tiny)
    ax.annotate(
        f"std ≈ {std[1]:.2e}\n(near-perfect — the GP is very sure here)",
        xy=(query_x[1], mean[1]),
        xytext=(query_x[1] + 0.4, mean[1] - 0.6),
        fontsize=9, ha="left",
        arrowprops=dict(arrowstyle="->", color="tab:blue", alpha=0.6),
    )

    ax.set_title("PyMC surrogate vs analytical truth (with training overlay)")
    ax.set_xlabel("a + b  (query and training points share this axis)")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"\nNumeric check (4 query points):")
    print(f"  predicted mean: {[float(f'{m:.4f}') for m in mean]}")
    print(f"  analytical y :  {[float(f'{t:.4f}') for t in query_truth]}")
    print(f"  std (each)   :  {[float(f'{s:.2e}') for s in std]}")


## Optional appendix: confidence check that the PyMC backend works

The cell below runs a single regression test from the package's own test suite. It's not part of the lesson — it's just a "is my PyMC install actually functional" smoke test you can run if anything earlier surprised you. Skip on first read.


In [ ]:
if not PYMC_AVAILABLE:
    print("Optional appendix SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    run_tool('pytest', '-q', 'tests/test_surrogate_backends.py', '-k', 'pymc_gp_backend_fit_sample_and_logprob')


## Recap: what T5 established

- A **surrogate** is a cheap probabilistic stand-in for an expensive model, trained
  from `sweep_rows.csv` — the same file T1 produced.
- A **GP** gives you a mean *and* a calibrated width. The width is the product; the
  mean alone is just interpolation.
- Predictive width **narrows near data and widens away from it**, and the mean
  reverts toward the prior outside the training range. That behaviour is the
  honest-extrapolation property you're paying for.
- Artifacts are **content-addressed and re-loadable** — T7 and T8 consume this one
  by reference rather than refitting.

One sentence to carry forward: *the surrogate's uncertainty is what makes it safe to
use in place of the model.*

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `PREFLIGHT: PyMC backend missing` | `pymc` absent from this kernel | `conda env create -f environment-pymc.yml`, then run this notebook on the `py312_bayesmm_pymc` kernel. The main env ships without backends by design. |
| Fit takes minutes, or PyTensor compiles C code | PyTensor is building its C ops | Set `PYTENSOR_FLAGS=cxx=` for the C-less path. Slower per-sample, far faster to start, and deterministic across machines. |
| `No surrogate artifact produced` in the self-check | Step 2's fit never ran | Scroll up for the preflight banner. Skipped steps say so loudly. |
| Predictions look flat / mean reverts everywhere | Too few training points, or the length-scale prior fights the data | This is real GP behaviour, not a bug — 9 points is deliberately marginal. Re-run T1's sweep with a denser grid and compare. |
| Predictive band is suspiciously narrow far from data | You're reading the mean, not the band | Confirm you plotted `std`; a GP is *supposed* to widen away from evidence. |

**Next:** T6 fits a fundamentally different surrogate — SBI's neural posterior
estimation — on this same data, so you can compare what a GP and a neural density
estimator each believe about the same 9 points.

## Final check: T5's PyMC surrogate is calibrated

Asserts the latest surrogate artifact for the T5 spec exists, has non-empty `posterior_draws` (the GP actually fit), and the predictive mean at the 4 query points matches the analytical truth `y = a + b` within absolute error 0.05. If this raises, the PyMC fit is broken or the surrogate dispatch is wrong.

In [ ]:
if not PYMC_AVAILABLE:
    print(f"\n[T5 self-check OK] PyMC steps skipped per preflight (PYMC_AVAILABLE={PYMC_AVAILABLE}).")
else:
    # Self-check: PyMC GP surrogate trained AND predicts close to analytical truth.
    from pathlib import Path as _P
    import json as _json
    import numpy as _np
    _arts = sorted((root / "tmp/surrogate_artifacts").glob("*/artifact.json"),
                   key=lambda p: p.stat().st_mtime)
    assert _arts, "No surrogate artifact produced — Step 2's `surrogate fit` didn't run."
    _latest = _arts[-1]
    _payload = _json.loads(_latest.read_text())
    assert _payload.get("backend") == "pymc_gp", f"Latest artifact is backend={_payload.get('backend')}, expected pymc_gp."
    # Re-eval on the same 4 query points used in Step 4 and check calibration.
    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate
    _spec = SurrogateSpec.model_validate(_json.loads((root / "tutorials/specs/surrogate.toy.pymc_gp.json").read_text()))
    _inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    _result = eval_surrogate(spec=_spec, inputs_payload=_inputs, n=200)
    _mean = _np.asarray(_result["summary"]["mean"], dtype=float)
    _truth = _np.asarray(_inputs["a"]) + _np.asarray(_inputs["b"])
    _mae = float(_np.mean(_np.abs(_mean - _truth)))
    assert _mae < 0.05, f"PyMC mean prediction MAE={_mae:.4f} > 0.05 — surrogate not calibrated for `y = a + b`."
    print(f"\n[T5 self-check OK] PyMC GP MAE={_mae:.5f} (< 0.05); artifact at {_latest.relative_to(root)}")
